In [2]:
#30 FEATURE
# gerekli kütüphaneleri yükledim
import pandas as pd
import numpy as np
import warnings
import os
import re
import matplotlib.pyplot as plt
import openpyxl

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_selection import SelectKBest, f_classif, mutual_info_classif
from mrmr import mrmr_classif

from sklearn.model_selection import (
    StratifiedKFold,
    StratifiedGroupKFold,
    cross_validate,
    GridSearchCV
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SkPipeline

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import (
    make_scorer, accuracy_score, f1_score, matthews_corrcoef,
    precision_score, recall_score, roc_auc_score
)

warnings.filterwarnings("ignore")


# ROC-AUC güvenli kısmı sorun çözücü şekilde
EPS = 1e-6

def _get_scores_from_estimator(estimator, X):
    if hasattr(estimator, "predict_proba"):
        proba = estimator.predict_proba(X)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(estimator, "decision_function"):
        s = estimator.decision_function(X)
        if np.ndim(s) > 1:
            s = s[:, 0]
        return s

    return estimator.predict(X)

def safe_roc_auc_estimator(estimator, X, y_true):
    try:
        y_score = _get_scores_from_estimator(estimator, X)
        auc = roc_auc_score(y_true, y_score)
        if np.isnan(auc):
            return 0.5
        if auc == 0.0:
            return EPS
        return float(auc)
    except Exception:
        return 0.5


# mRMR için olan eklentiler
class MRMRSelector(BaseEstimator, TransformerMixin):
    def __init__(self, K=30):
        self.K = K
        self.selected_features_ = None
        self.selected_indices_ = None
        self.feature_names_in_ = None

    def fit(self, X, y):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy().reset_index(drop=True)
            self.feature_names_in_ = list(X_df.columns)
        else:
            X_df = pd.DataFrame(
                X,
                columns=[f"f{i}" for i in range(X.shape[1])]
            ).reset_index(drop=True)
            self.feature_names_in_ = list(X_df.columns)

        y_sr = pd.Series(y).reset_index(drop=True)

        if len(X_df) != len(y_sr):
            raise ValueError(
                f"X ve y örnek sayıları eşit değil: "
                f"X={len(X_df)}, y={len(y_sr)}"
            )

        self.selected_features_ = mrmr_classif(
            X=X_df,
            y=y_sr,
            K=self.K
        )

        self.selected_indices_ = [
            X_df.columns.get_loc(column)
            for column in self.selected_features_
        ]

        return self

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            X_df = X.copy()
        else:
            X_df = pd.DataFrame(
                X,
                columns=self.feature_names_in_
            )

        return X_df[self.selected_features_]

    def transform(self, X):
        if isinstance(X, pd.DataFrame):
            return X[self.selected_features_]
        else:
            X_df = pd.DataFrame(X, columns=self.feature_names_in_)
            return X_df[self.selected_features_]
          


# TOPSIS Analiz için ek şeyler
def topsis_analysis(results_df, metric_columns, weights):
    """
    Verilen bir DataFrame'e TOPSIS analizi uygular ve sıralanmış sonuçları döndürür.
    Sıfıra bölme hatalarına karşı korumalıdır.
    """
    print("--- TOPSIS Analizi Başlatılıyor ---")

    decision_matrix = results_df[metric_columns].apply(pd.to_numeric, errors='coerce').fillna(0).values

    denominator = np.sqrt(np.sum(decision_matrix**2, axis=0)) + 1e-8
    norm_matrix = decision_matrix / denominator

    weighted_matrix = norm_matrix * np.array(weights)

    ideal_best = np.max(weighted_matrix, axis=0)
    ideal_worst = np.min(weighted_matrix, axis=0)

    dist_to_best = np.sqrt(np.sum((weighted_matrix - ideal_best)**2, axis=1))
    dist_to_worst = np.sqrt(np.sum((weighted_matrix - ideal_worst)**2, axis=1))

    topsis_score = dist_to_worst / (dist_to_best + dist_to_worst + 1e-8)

    ranked_df = results_df.copy()
    ranked_df['Topsis_Score'] = topsis_score
    ranked_df['Rank'] = ranked_df['Topsis_Score'].rank(ascending=False, method='min').astype(int)

    return ranked_df.sort_values(by='Rank', ascending=True).reset_index(drop=True)


# EKLENDİ:
# Seçilen miRNA'ların tümörde mi normalde mi daha yüksek olduğunu hesaplar
def get_biomarker_direction_for_selected_mirnas(X, y, selected_mirnas, tumor_label=1, normal_label=0):
    """
    Seçilen miRNA'ların tümörde mi normalde mi daha yüksek olduğunu hesaplar.

    tumor_label = 1  (TP)
    normal_label = 0 (NT)

    Veri log2(RPM+1) olduğu için:
    log2FC_Tumor_vs_Normal = mean(tumor) - mean(normal)
    """
    rows = []

    y = pd.Series(y).reset_index(drop=True)
    X = X.reset_index(drop=True)

    for mirna in selected_mirnas:
        if mirna not in X.columns:
            continue

        tumor_vals = X.loc[y == tumor_label, mirna]
        normal_vals = X.loc[y == normal_label, mirna]

        tumor_mean = tumor_vals.mean()
        normal_mean = normal_vals.mean()

        log2fc = tumor_mean - normal_mean

        if log2fc > 0:
            direction = "Up in Tumor"
            biomarker_type = "Positive biomarker candidate"
            interpretation = "Higher expression in tumor than normal"
        elif log2fc < 0:
            direction = "Down in Tumor"
            biomarker_type = "Negative biomarker candidate"
            interpretation = "Lower expression in tumor than normal"
        else:
            direction = "No Change"
            biomarker_type = "No clear biomarker direction"
            interpretation = "Similar expression in tumor and normal"

        rows.append({
            "miRNA": mirna,
            "Tumor_mean": tumor_mean,
            "Normal_mean": normal_mean,
            "log2FC_Tumor_vs_Normal": log2fc,
            "Direction": direction,
            "Biomarker_Type": biomarker_type,
            "Interpretation": interpretation
        })

    biomarker_df = pd.DataFrame(rows)
    if not biomarker_df.empty:
        biomarker_df = biomarker_df.sort_values("log2FC_Tumor_vs_Normal", ascending=False).reset_index(drop=True)
    return biomarker_df


# Dosya yolu ve parametreleri girdim
file_path = "THCA_mature_miRNA_3p5p_unique_patients.csv"

param_grids = {
    "SVM": {
        'classifier__C': [0.1, 1, 10],
        'classifier__kernel': ['linear', 'rbf']
    },
    "Log. Reg.": {
        'classifier__C': [0.1, 1, 10],
        'classifier__penalty': ['l1', 'l2']
    },
    "Extra Trees": {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "LDA": [
        {
            'classifier__solver': ['svd']
        },
        {
            'classifier__solver': ['lsqr', 'eigen'],
            'classifier__shrinkage': [None, 'auto']
        }
    ],
    "Random Forest": {
        'classifier__n_estimators': [50, 100],
        'classifier__max_depth': [None, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "Gradient Boosting": {
        'classifier__n_estimators': [50, 100],
        'classifier__learning_rate': [0.05, 0.1],
        'classifier__max_depth': [2, 3]
    },
    "SGD": [
        {
            'classifier__alpha': [0.0001, 0.001, 0.01],
            'classifier__penalty': ['l1', 'l2']
        },
        {
            'classifier__alpha': [0.0001, 0.001, 0.01],
            'classifier__penalty': ['elasticnet'],
            'classifier__l1_ratio': [0.15, 0.5]
        }
    ],
    "KNN": {
        'classifier__n_neighbors': [3, 5, 7],
        'classifier__weights': ['uniform', 'distance'],
        'classifier__p': [1, 2]
    },
    "Decision Tree": {
        'classifier__criterion': ['gini', 'entropy'],
        'classifier__max_depth': [None, 5, 10, 20],
        'classifier__min_samples_split': [2, 5]
    },
    "AdaBoost": {
        'classifier__n_estimators': [50, 100, 200],
        'classifier__learning_rate': [0.01, 0.1, 1.0]
    }
}

# Low-expression filtering sonrasında 30 mature-miRNA feature seçilir
K_FEATURES = 30
FEATURE_OUTPUT_EXCEL = "selected_features_by_scenario_30.xlsx"
PLOT_DIR = "selected_feature_plots_30"
RESULTS_EXCEL = "classification_results_leakage_fixed_30.xlsx"

TOPSIS_OUTPUT_EXCEL = "topsis_sonuclari_sirali_30.xlsx"
TOPSIS_SHEET_NAME = "Tum Sonuçlar (Sirali)"
METRICS_TO_EVALUATE = ['MCC', 'F1-Score', 'ROC AUC', 'Accuracy', 'Precision', 'Recall']
METRIC_WEIGHTS = [0.30, 0.30, 0.15, 0.10, 0.075, 0.075]


# smote undersampler ve unbalanced olayı burda
def make_safe_sheet_name(name, max_len=31):
    name = re.sub(r'[\\/*?:\[\]]', '_', name)
    return name[:max_len]

def scenario_code(data_handling, optimization_status, selector_name):
    handling_map = {
        "Unbalanced": "UB",
        "Downsampled": "DS",
        "Oversampled (SMOTE)": "OS"
    }
    opt_map = {
        "Unoptimized": "U",
        "Optimized": "O"
    }
    return f"{handling_map.get(data_handling, 'X')}_{opt_map.get(optimization_status, 'X')}_{selector_name}"

def get_selector(selector_name, k=30):
    if selector_name == "ANOVA":
        return SelectKBest(score_func=f_classif, k=k)
    elif selector_name == "MI":
        return SelectKBest(score_func=mutual_info_classif, k=k)
    elif selector_name == "mRMR":
        return MRMRSelector(K=k)
    else:
        raise ValueError(f"Bilinmeyen selector: {selector_name}")

def preprocess_for_selector(
    X,
    y,
    data_handling,
    random_state=42
):
    """
    Pipeline'ın selector öncesi kısmını taklit eder:
    scaler -> opsiyonel downsampling veya SMOTE.

    Bu fonksiyon feature incelemesinde yalnızca ilgili
    training fold üzerinde çağrılır.
    """

    X = X.copy().reset_index(drop=True)
    y_out = pd.Series(y).reset_index(drop=True)

    if len(X) != len(y_out):
        raise ValueError(
            f"X ve y örnek sayıları eşit değil: "
            f"X={len(X)}, y={len(y_out)}"
        )

    scaler = StandardScaler()

    X_scaled = pd.DataFrame(
        scaler.fit_transform(X),
        columns=X.columns
    ).reset_index(drop=True)

    if data_handling == "Downsampled":
        undersampler = RandomUnderSampler(
            random_state=random_state
        )

        X_res, y_res = undersampler.fit_resample(
            X_scaled,
            y_out
        )

        X_res = pd.DataFrame(
            X_res,
            columns=X.columns
        ).reset_index(drop=True)

        y_res = pd.Series(
            y_res
        ).reset_index(drop=True)

        return X_res, y_res

    if data_handling == "Oversampled (SMOTE)":
        smote = SMOTE(
            random_state=random_state
        )

        X_res, y_res = smote.fit_resample(
            X_scaled,
            y_out
        )

        X_res = pd.DataFrame(
            X_res,
            columns=X.columns
        ).reset_index(drop=True)

        y_res = pd.Series(
            y_res
        ).reset_index(drop=True)

        return X_res, y_res

    return X_scaled, y_out

def get_selected_features_from_selector(selector, X_columns):
    if isinstance(selector, MRMRSelector):
        selected = list(selector.selected_features_)
        scores_df = pd.DataFrame({
            "miRNA": selected,
            "score_or_rank": list(range(1, len(selected) + 1))
        })
        return selected, scores_df

    support = selector.get_support()
    selected = list(pd.Index(X_columns)[support])

    if hasattr(selector, "scores_") and selector.scores_ is not None:
        scores = selector.scores_
        score_map = pd.DataFrame({
            "miRNA": X_columns,
            "raw_score": scores
        })
        score_map = score_map[score_map["miRNA"].isin(selected)].copy()
        score_map["score_or_rank"] = score_map["raw_score"]
        score_map = score_map.sort_values("score_or_rank", ascending=False).reset_index(drop=True)
        return list(score_map["miRNA"]), score_map[["miRNA", "score_or_rank"]]

    scores_df = pd.DataFrame({
        "miRNA": selected,
        "score_or_rank": list(range(1, len(selected) + 1))
    })
    return selected, scores_df

def inspect_selected_features_for_scenario(
    X, y, groups, data_handling, selector_name, optimization_status, k=30, cv_splits=5, random_state=42
):
    """
    1) Tüm veri üzerinde seçilen feature'lar
    2) CV fold'larında seçilme sıklıkları
    """
    X_proc, y_proc = preprocess_for_selector(X, y, data_handling, random_state=random_state)
    selector_full = get_selector(selector_name, k=k)
    selector_full.fit(X_proc, y_proc)
    selected_full, selected_scores_df = get_selected_features_from_selector(selector_full, X_proc.columns)

    full_selected_df = selected_scores_df.copy()
    full_selected_df.insert(0, "data_handling", data_handling)
    full_selected_df.insert(1, "optimization", optimization_status)
    full_selected_df.insert(2, "selector", selector_name)

    groups = pd.Series(groups).reset_index(drop=True)

    cv = StratifiedGroupKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)
    freq_counter = pd.Series(0, index=X.columns, dtype=int)

    fold_details = []

    for fold_id, (train_idx, test_idx) in enumerate(cv.split(X, y, groups), start=1):
        X_train = X.iloc[train_idx].copy()
        y_train = y.iloc[train_idx].copy()

        X_train_proc, y_train_proc = preprocess_for_selector(X_train, y_train, data_handling, random_state=random_state)
        selector_fold = get_selector(selector_name, k=k)
        selector_fold.fit(X_train_proc, y_train_proc)

        selected_fold, _ = get_selected_features_from_selector(selector_fold, X_train_proc.columns)

        freq_counter.loc[selected_fold] += 1

        for feat in selected_fold:
            fold_details.append({
                "fold": fold_id,
                "miRNA": feat,
                "selected": 1
            })

    freq_df = freq_counter.reset_index()
    freq_df.columns = ["miRNA", "selection_count"]
    freq_df["selection_frequency"] = freq_df["selection_count"] / cv_splits
    freq_df = freq_df.sort_values(["selection_count", "miRNA"], ascending=[False, True]).reset_index(drop=True)
    freq_df.insert(0, "data_handling", data_handling)
    freq_df.insert(1, "optimization", optimization_status)
    freq_df.insert(2, "selector", selector_name)

    fold_detail_df = pd.DataFrame(fold_details)
    if not fold_detail_df.empty:
        fold_detail_df.insert(0, "data_handling", data_handling)
        fold_detail_df.insert(1, "optimization", optimization_status)
        fold_detail_df.insert(2, "selector", selector_name)

    return full_selected_df, freq_df, fold_detail_df

def save_top_frequency_plot(freq_df, title, out_path, top_n=30):
    plot_df = freq_df.head(top_n).copy()
    if plot_df.empty:
        return

    plt.figure(figsize=(12, 8))
    plt.barh(plot_df["miRNA"][::-1], plot_df["selection_count"][::-1])
    plt.xlabel("Selection count across CV folds")
    plt.ylabel("miRNA")
    plt.title(title)
    plt.tight_layout()
    plt.savefig(out_path, dpi=300, bbox_inches='tight')
    plt.close()


# Classifierlar
def get_classifiers(optimization_status):
    classifiers_base = {
        "SVM": SVC(gamma='auto', probability=True, random_state=42),
        "Log. Reg.": LogisticRegression(solver='liblinear', random_state=42),
        "Extra Trees": ExtraTreesClassifier(n_estimators=100, random_state=42),
        "LDA": LinearDiscriminantAnalysis(),
        "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
        "SGD": SGDClassifier(loss='log_loss', random_state=42),
        "KNN": KNeighborsClassifier(),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42)
    }

    if optimization_status == "Unoptimized":
        return classifiers_base

    optimized_classifiers = {}
    for name, classifier in classifiers_base.items():
        optimized_classifiers[name] = classifier
    return optimized_classifiers


# Skorlama
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'f1_score': make_scorer(f1_score, zero_division=0),
    'mcc_score': make_scorer(matthews_corrcoef),
    'precision_score': make_scorer(precision_score, zero_division=0),
    'recall_score': make_scorer(recall_score, zero_division=0),
    'roc_auc_score': safe_roc_auc_estimator
}


def calculate_fold_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = _get_scores_from_estimator(model, X_test)

    try:
        auc = roc_auc_score(y_test, y_score)
        if np.isnan(auc):
            auc = 0.5
        elif auc == 0.0:
            auc = EPS
    except Exception:
        auc = 0.5

    return {
        "Accuracy": accuracy_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred, zero_division=0),
        "MCC": matthews_corrcoef(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, zero_division=0),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "ROC AUC": float(auc)
    }


# Pipeline kurma
def build_pipeline(classifier, selector_name, data_handling, optimization_status, k=30):
    selector = get_selector(selector_name, k=k)

    if data_handling == "Oversampled (SMOTE)":
        base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('smote', SMOTE(random_state=42)),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])
    elif data_handling == "Downsampled":
        base_pipeline = ImbPipeline([
            ('scaler', StandardScaler()),
            ('downsampler', RandomUnderSampler(random_state=42)),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])
    else:
        base_pipeline = SkPipeline([
            ('scaler', StandardScaler()),
            ('feature_selection', selector),
            ('classifier', classifier)
        ])

    return base_pipeline


# classifierları GridSearch ile optimize ettim
def build_optimized_model(classifier_name, classifier, selector_name, data_handling, k=30):
    pipeline = build_pipeline(
        classifier=classifier,
        selector_name=selector_name,
        data_handling=data_handling,
        optimization_status="Optimized",
        k=k
    )

    grid = param_grids[classifier_name]

    inner_cv = StratifiedGroupKFold(
        n_splits=3,
        shuffle=True,
        random_state=42
    )

    gs = GridSearchCV(
        estimator=pipeline,
        param_grid=grid,
        cv=inner_cv,
        scoring='f1',
        n_jobs=1,
        error_score=0.0
    )
    return gs


# Ana değerlendirme fonksiyonu
def evaluate_dataset(X, y, groups, title, data_handling, selector_name, optimization_status, k=30):
    print(f"\n--- {title} | FS: {selector_name} | Opt: {optimization_status} ---")

    X = X.reset_index(drop=True)
    y = pd.Series(y).reset_index(drop=True)
    groups = pd.Series(groups).reset_index(drop=True)

    classifiers = get_classifiers(optimization_status)
    outer_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    results = {}

    for clf_name, clf in classifiers.items():
        try:
            if optimization_status == "Optimized":
                fold_metrics = []

                for fold_id, (train_idx, test_idx) in enumerate(
                    outer_cv.split(X, y, groups),
                    start=1
                ):
                    X_train = X.iloc[train_idx].reset_index(drop=True)
                    X_test = X.iloc[test_idx].reset_index(drop=True)
                    y_train = y.iloc[train_idx].reset_index(drop=True)
                    y_test = y.iloc[test_idx].reset_index(drop=True)
                    groups_train = groups.iloc[train_idx].reset_index(drop=True)
                    groups_test = groups.iloc[test_idx].reset_index(drop=True)

                    overlap = set(groups_train).intersection(set(groups_test))
                    if overlap:
                        raise ValueError(
                            f"Fold {fold_id}: train ve test arasında "
                            f"{len(overlap)} ortak hasta bulundu."
                        )

                    model = build_optimized_model(
                        classifier_name=clf_name,
                        classifier=clf,
                        selector_name=selector_name,
                        data_handling=data_handling,
                        k=k
                    )

                    model.fit(
                        X_train,
                        y_train,
                        groups=groups_train
                    )

                    fold_metrics.append(
                        calculate_fold_metrics(
                            model=model,
                            X_test=X_test,
                            y_test=y_test
                        )
                    )

                fold_metrics_df = pd.DataFrame(fold_metrics)

                results[clf_name] = {
                    "Accuracy": fold_metrics_df["Accuracy"].mean(),
                    "F1-Score": fold_metrics_df["F1-Score"].mean(),
                    "MCC": fold_metrics_df["MCC"].mean(),
                    "Precision": fold_metrics_df["Precision"].mean(),
                    "Recall": fold_metrics_df["Recall"].mean(),
                    "ROC AUC": fold_metrics_df["ROC AUC"].mean()
                }

            else:
                model = build_pipeline(
                    classifier=clf,
                    selector_name=selector_name,
                    data_handling=data_handling,
                    optimization_status=optimization_status,
                    k=k
                )

                cv_results = cross_validate(
                    model,
                    X,
                    y,
                    groups=groups,
                    cv=outer_cv,
                    scoring=scoring,
                    n_jobs=-1,
                    error_score=0.0
                )

                results[clf_name] = {
                    "Accuracy": cv_results['test_accuracy'].mean(),
                    "F1-Score": cv_results['test_f1_score'].mean(),
                    "MCC": cv_results['test_mcc_score'].mean(),
                    "Precision": cv_results['test_precision_score'].mean(),
                    "Recall": cv_results['test_recall_score'].mean(),
                    "ROC AUC": cv_results['test_roc_auc_score'].mean()
                }

        except Exception as e:
            print(f"Hata ({clf_name}): {e}")
            results[clf_name] = {
                "Accuracy": 0.0,
                "F1-Score": 0.0,
                "MCC": 0.0,
                "Precision": 0.0,
                "Recall": 0.0,
                "ROC AUC": 0.0
            }

    results_df = pd.DataFrame(results).T.round(3)
    print(results_df)
    return results_df


# Downsample helper
def create_downsampled_df(df, label_col='label', majority_class=1, minority_class=0, random_state=42):
    df_majority = df[df[label_col] == majority_class]
    df_minority = df[df[label_col] == minority_class]

    n_min = len(df_minority)
    df_down = pd.concat([
        df_majority.sample(n=n_min, random_state=random_state),
        df_minority
    ]).sample(frac=1, random_state=random_state).reset_index(drop=True)

    return df_down


# Ana blok
try:
    os.makedirs(PLOT_DIR, exist_ok=True)

    df = pd.read_csv(file_path)

    if 'label' not in df.columns:
        raise KeyError("Veri setinde 'label' kolonu bulunamadı.")

    if 'Sample' not in df.columns:
        raise KeyError("Veri setinde 'Sample' kolonu bulunamadı.")

    # ============================================================
    # NORMALİZASYON BLOĞU (raw count -> RPM -> log2(RPM+1))
    # ============================================================
    feature_cols = [
        col for col in df.columns
        if col not in ['label', 'Sample']
    ]

    df[feature_cols] = df[feature_cols].apply(pd.to_numeric, errors='coerce').fillna(0.0)
    df[feature_cols] = df[feature_cols].clip(lower=0)

    lib_sizes = df[feature_cols].sum(axis=1)
    lib_sizes = lib_sizes.replace(0, np.nan)

    rpm = df[feature_cols].div(lib_sizes, axis=0) * 1e6
    rpm = rpm.fillna(0.0)

    log_rpm = np.log2(rpm + 1.0)

    # ============================================================
    # LOW-EXPRESSION FILTERING
    # En az %20 sample'da RPM > 1 olan miRNA'ları tut
    # ============================================================
    min_sample_fraction = 0.20
    min_samples_required = int(np.ceil(rpm.shape[0] * min_sample_fraction))

    keep_mask = (rpm > 1).sum(axis=0) >= min_samples_required
    kept_features = keep_mask[keep_mask].index.tolist()

    if len(kept_features) == 0:
        raise ValueError("Low-expression filtering sonrası hiç miRNA kalmadı. Eşiği gevşetmelisin.")

    print(f"Toplam başlangıç feature sayısı: {len(feature_cols)}")
    print(f"Filtre sonrası kalan feature sayısı: {len(kept_features)}")
    print(f"Eşik: en az {min_samples_required} sample'da RPM > 1")

    df = pd.concat(
        [
            df[['Sample', 'label']].reset_index(drop=True),
            log_rpm[kept_features].reset_index(drop=True)
        ],
        axis=1
    )

    groups_full = df['Sample'].astype(str).str[:12].reset_index(drop=True)
    X_full = df.drop(['Sample', 'label'], axis=1).reset_index(drop=True)
    y_full = df['label'].reset_index(drop=True)

    all_results = []

    # 1) Unbalanced
    print("\n" + "=" * 60)
    print("STARTING UNBALANCED DATA ANALYSIS")
    print("=" * 60)

    for selector_name in ['ANOVA', 'MI', 'mRMR']:
        for opt_status in ['Unoptimized', 'Optimized']:
            res = evaluate_dataset(
                X=X_full,
                y=y_full,
                groups=groups_full,
                title="Unbalanced Data",
                data_handling="Unbalanced",
                selector_name=selector_name,
                optimization_status=opt_status,
                k=K_FEATURES
            )
            res['Data Handling'] = 'Unbalanced'
            res['Feature Selector'] = selector_name
            res['Optimization'] = opt_status
            all_results.append(res)

    # 2) Downsampled
    print("\n" + "=" * 60)
    print("STARTING DOWNSAMPLED DATA ANALYSIS")
    print("=" * 60)

    for selector_name in ['ANOVA', 'MI', 'mRMR']:
        for opt_status in ['Unoptimized', 'Optimized']:
            res = evaluate_dataset(
                X=X_full,
                y=y_full,
                groups=groups_full,
                title="Downsampled Data",
                data_handling="Downsampled",
                selector_name=selector_name,
                optimization_status=opt_status,
                k=K_FEATURES
            )
            res['Data Handling'] = 'Downsampled'
            res['Feature Selector'] = selector_name
            res['Optimization'] = opt_status
            all_results.append(res)

    # 3) SMOTE
    print("\n" + "=" * 60)
    print("STARTING SMOTE WITHIN-CV ANALYSIS")
    print("=" * 60)

    if y_full.value_counts().min() > 1:
        for selector_name in ['ANOVA', 'MI', 'mRMR']:
            for opt_status in ['Unoptimized', 'Optimized']:
                res = evaluate_dataset(
                    X=X_full,
                    y=y_full,
                    groups=groups_full,
                    title="Oversampled Data",
                    data_handling="Oversampled (SMOTE)",
                    selector_name=selector_name,
                    optimization_status=opt_status,
                    k=K_FEATURES
                )
                res['Data Handling'] = 'Oversampled (SMOTE)'
                res['Feature Selector'] = selector_name
                res['Optimization'] = opt_status
                all_results.append(res)
    else:
        print("SMOTE analizi atlandı: Azınlık sınıfı çok küçük.")

    # ============================================================
    # Seçilen feature'ları çıkar ve grafiklerini oluştur
    # ============================================================
    scenario_datasets = [
        ("Unbalanced", "Unoptimized", X_full, y_full),
        ("Unbalanced", "Optimized", X_full, y_full),
        ("Downsampled", "Unoptimized", X_full, y_full),
        ("Downsampled", "Optimized", X_full, y_full)
    ]

    if y_full.value_counts().min() > 1:
        scenario_datasets.extend([
            ("Oversampled (SMOTE)", "Unoptimized", X_full, y_full),
            ("Oversampled (SMOTE)", "Optimized", X_full, y_full)
        ])

    # EKLENDİ:
    # Her senaryonun hangi dataset ile çalıştığını sakla
    scenario_dataset_lookup = {
        ("Unbalanced", "Unoptimized"): (X_full, y_full),
        ("Unbalanced", "Optimized"): (X_full, y_full),
        ("Downsampled", "Unoptimized"): (X_full, y_full),
        ("Downsampled", "Optimized"): (X_full, y_full)
    }

    if y_full.value_counts().min() > 1:
        scenario_dataset_lookup.update({
            ("Oversampled (SMOTE)", "Unoptimized"): (X_full, y_full),
            ("Oversampled (SMOTE)", "Optimized"): (X_full, y_full)
        })

    scenario_feature_map = {}

    with pd.ExcelWriter(FEATURE_OUTPUT_EXCEL, engine='openpyxl') as feat_writer:
        for data_handling, optimization_status, X_use, y_use in scenario_datasets:
            for selector_name in ['ANOVA', 'MI', 'mRMR']:
                full_selected_df, freq_df, fold_detail_df = inspect_selected_features_for_scenario(
                    X=X_use,
                    y=y_use,
                    groups=groups_full,
                    data_handling=data_handling,
                    selector_name=selector_name,
                    optimization_status=optimization_status,
                    k=K_FEATURES,
                    cv_splits=5,
                    random_state=42
                )

                code = scenario_code(data_handling, optimization_status, selector_name)

                full_sheet = make_safe_sheet_name(f"{code}_full")
                freq_sheet = make_safe_sheet_name(f"{code}_freq")
                fold_sheet = make_safe_sheet_name(f"{code}_folds")

                full_selected_df.to_excel(feat_writer, sheet_name=full_sheet, index=False)
                freq_df.to_excel(feat_writer, sheet_name=freq_sheet, index=False)

                if not fold_detail_df.empty:
                    fold_detail_df.to_excel(feat_writer, sheet_name=fold_sheet, index=False)

                plot_file = os.path.join(PLOT_DIR, f"{code}_top_selected.png")
                plot_title = f"{data_handling} | {optimization_status} | {selector_name}\nTop miRNA selection frequency"
                save_top_frequency_plot(freq_df, plot_title, plot_file, top_n=30)

                scenario_feature_map[(data_handling, optimization_status, selector_name)] = {
                    "full_selected_df": full_selected_df.copy(),
                    "freq_df": freq_df.copy(),
                    "code": code
                }

    # ============================================================
    # Performans sonuçlarını yaz
    # ============================================================
    final_results_df = None
    if all_results:
        final_results_df = pd.concat(all_results).reset_index().rename(columns={'index': 'Classifier'})

        with pd.ExcelWriter(RESULTS_EXCEL, engine='openpyxl') as writer:
            all_sorted = final_results_df.sort_values(
                by=['Data Handling', 'Optimization', 'Feature Selector', 'MCC'],
                ascending=[True, True, True, False]
            )
            all_sorted.to_excel(writer, sheet_name=TOPSIS_SHEET_NAME, index=False)

            for handling_type in final_results_df['Data Handling'].unique():
                for opt_status in final_results_df['Optimization'].unique():
                    sheet_name = f"{handling_type}-{opt_status}"[:31]
                    subset_df = final_results_df[
                        (final_results_df['Data Handling'] == handling_type) &
                        (final_results_df['Optimization'] == opt_status)
                    ].copy()

                    subset_sorted = subset_df.sort_values(
                        by=['Feature Selector', 'MCC'],
                        ascending=[True, False]
                    )
                    subset_sorted.to_excel(writer, sheet_name=sheet_name, index=False)

    # ============================================================
    # TOPSIS Analizi
    # ============================================================
    if final_results_df is not None and not final_results_df.empty:
        final_ranked_results = topsis_analysis(final_results_df, METRICS_TO_EVALUATE, METRIC_WEIGHTS)

        optimized_only = final_ranked_results[final_ranked_results["Optimization"] == "Optimized"].copy()
        best_optimized_row = None
        best_optimized_features_df = pd.DataFrame()
        best_optimized_freq_df = pd.DataFrame()
        best_optimized_summary_df = pd.DataFrame()
        best_optimized_biomarker_df = pd.DataFrame()

        if not optimized_only.empty:
            best_optimized_row = optimized_only.sort_values("Rank", ascending=True).iloc[0]

            best_key = (
                best_optimized_row["Data Handling"],
                best_optimized_row["Optimization"],
                best_optimized_row["Feature Selector"]
            )

            if best_key in scenario_feature_map:
                best_optimized_features_df = scenario_feature_map[best_key]["full_selected_df"].copy()
                best_optimized_freq_df = scenario_feature_map[best_key]["freq_df"].copy()

                # EKLENDİ:
                # En iyi optimized senaryonun seçtiği miRNA'ların yönünü hesapla
                selected_mirnas = best_optimized_features_df["miRNA"].tolist()

                # Ekspresyon yönü tüm orijinal normalize edilmiş veri üzerinden hesaplanır.
                # Downsampling yalnızca model eğitimi için kullanılır.
                best_optimized_biomarker_df = get_biomarker_direction_for_selected_mirnas(
                    X=X_full,
                    y=y_full,
                    selected_mirnas=selected_mirnas,
                    tumor_label=1,
                    normal_label=0
                )

                # Aynı feature tablosuna biomarker yön bilgisini ekle
                if not best_optimized_biomarker_df.empty:
                    best_optimized_features_df = best_optimized_features_df.merge(
                        best_optimized_biomarker_df,
                        on="miRNA",
                        how="left"
                    )

                best_optimized_summary_df = pd.DataFrame([{
                    "Best_Optimized_Classifier": best_optimized_row["Classifier"],
                    "Best_Optimized_Data_Handling": best_optimized_row["Data Handling"],
                    "Best_Optimized_Optimization": best_optimized_row["Optimization"],
                    "Best_Optimized_Feature_Selector": best_optimized_row["Feature Selector"],
                    "Best_Optimized_Topsis_Score": best_optimized_row["Topsis_Score"],
                    "Best_Optimized_Rank": best_optimized_row["Rank"],
                    "MCC": best_optimized_row["MCC"],
                    "F1-Score": best_optimized_row["F1-Score"],
                    "ROC AUC": best_optimized_row["ROC AUC"],
                    "Accuracy": best_optimized_row["Accuracy"],
                    "Precision": best_optimized_row["Precision"],
                    "Recall": best_optimized_row["Recall"]
                }])

        with pd.ExcelWriter(TOPSIS_OUTPUT_EXCEL, engine='openpyxl') as topsis_writer:
            final_ranked_results.to_excel(topsis_writer, sheet_name="TOPSIS_Sirali_30", index=False)

            if not optimized_only.empty:
                optimized_only.to_excel(topsis_writer, sheet_name="TOPSIS_Optimized_30", index=False)

            if not best_optimized_summary_df.empty:
                best_optimized_summary_df.to_excel(topsis_writer, sheet_name="En_Iyi_Optimized_30", index=False)

            if not best_optimized_features_df.empty:
                best_optimized_features_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_miRNA_30",
                    index=False
                )

            if not best_optimized_freq_df.empty:
                best_optimized_freq_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_Freq_30",
                    index=False
                )

            # EKLENDİ:
            # Biomarker yön bilgisini ayrı sheet olarak da kaydet
            if not best_optimized_biomarker_df.empty:
                best_optimized_biomarker_df.to_excel(
                    topsis_writer,
                    sheet_name="En_Iyi_Optimized_Biomarker_30",
                    index=False
                )

        print("\n" + "=" * 60)
        print("🏆 TOPSIS ANALİZİ BAŞARIYLA TAMAMLANDI 🏆")
        print("=" * 60)
        print("\nEn iyi 10 model kombinasyonu:\n")
        print(final_ranked_results.head(10))

        if best_optimized_row is not None:
            print("\nEn iyi OPTIMIZED senaryo:")
            print(
                f"Classifier={best_optimized_row['Classifier']} | "
                f"Data Handling={best_optimized_row['Data Handling']} | "
                f"Selector={best_optimized_row['Feature Selector']} | "
                f"Topsis Score={best_optimized_row['Topsis_Score']:.4f}"
            )
            print(f"Bu senaryonun seçtiği miRNA'lar '{TOPSIS_OUTPUT_EXCEL}' içindeki 'En_Iyi_Optimized_miRNA' sayfasına yazıldı.")
            print(f"Biyomarker yön bilgileri '{TOPSIS_OUTPUT_EXCEL}' içindeki 'En_Iyi_Optimized_Biomarker' sayfasına yazıldı.")

    print("\n" + "=" * 60)
    print(f"TÜM SONUÇLAR BAŞARIYLA '{RESULTS_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print(f"SEÇİLEN FEATURE'LAR BAŞARIYLA '{FEATURE_OUTPUT_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print(f"GRAFİKLER '{PLOT_DIR}' klasörüne kaydedildi.")
    print(f"TOPSIS SONUÇLARI BAŞARIYLA '{TOPSIS_OUTPUT_EXCEL}' DOSYASINA KAYDEDİLDİ.")
    print("=" * 60)

except FileNotFoundError:
    print(f"❌ HATA: '{file_path}' veya ara çıktı dosyalarından biri bulunamadı.")
except KeyError as e:
    print(f"❌ HATA: {e}")
    
except ValueError as e:
    print(f"❌ HATA: {e}")
except Exception as e:
    print(f"Beklenmedik bir hata oluştu: {e}")

Toplam başlangıç feature sayısı: 2202
Filtre sonrası kalan feature sayısı: 533
Eşik: en az 113 sample'da RPM > 1

STARTING UNBALANCED DATA ANALYSIS

--- Unbalanced Data | FS: ANOVA | Opt: Unoptimized ---
                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.980     0.989  0.899      0.992   0.986    0.990
Log. Reg.             0.984     0.991  0.922      0.996   0.986    0.993
Extra Trees           0.984     0.991  0.917      0.994   0.988    0.995
LDA                   0.982     0.990  0.913      0.996   0.984    0.985
Random Forest         0.980     0.989  0.895      0.986   0.992    0.995
Gradient Boosting     0.980     0.989  0.893      0.986   0.992    0.976
SGD                   0.979     0.988  0.889      0.988   0.988    0.986
KNN                   0.979     0.988  0.899      0.998   0.978    0.984
Decision Tree         0.968     0.982  0.823      0.979   0.986    0.898
AdaBoost              0.973     0.985  0.860      0.986   0.984   

100%|██████████| 30/30 [00:02<00:00, 13.16it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.984     0.991  0.918      0.994   0.988    0.988
Log. Reg.             0.986     0.992  0.928      0.996   0.988    0.993
Extra Trees           0.984     0.991  0.916      0.990   0.992    0.986
LDA                   0.980     0.989  0.908      0.998   0.980    0.989
Random Forest         0.986     0.992  0.922      0.990   0.994    0.996
Gradient Boosting     0.980     0.989  0.895      0.988   0.990    0.993
SGD                   0.975     0.986  0.867      0.986   0.986    0.976
KNN                   0.980     0.989  0.906      0.996   0.982    0.985
Decision Tree         0.965     0.980  0.809      0.977   0.984    0.889
AdaBoost              0.982     0.990  0.905      0.988   0.992    0.992

--- Unbalanced Data | FS: mRMR | Opt: Optimized ---


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:01<00:00, 29.61it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.982     0.990  0.906      0.990   0.990    0.988
Log. Reg.             0.982     0.990  0.908      0.992   0.988    0.993
Extra Trees           0.986     0.992  0.926      0.992   0.992    0.986
LDA                   0.980     0.989  0.908      0.998   0.980    0.989
Random Forest         0.986     0.992  0.922      0.990   0.994    0.996
Gradient Boosting     0.982     0.990  0.904      0.988   0.992    0.995
SGD                   0.979     0.988  0.883      0.984   0.992    0.992
KNN                   0.979     0.988  0.897      0.996   0.980    0.985
Decision Tree         0.966     0.981  0.823      0.982   0.980    0.914
AdaBoost              0.984     0.991  0.914      0.988   0.994    0.994

STARTING DOWNSAMPLED DATA ANALYSIS

--- Downsampled Data | FS: ANOVA | Opt: Unoptimized ---
                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.972    

100%|██████████| 30/30 [00:02<00:00, 14.46it/s]


... (5 outputs left out)

100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 30.13it/s]


                   Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC
SVM                   0.984     0.991  0.919      0.996   0.986    0.989
Log. Reg.             0.979     0.988  0.900      0.998   0.978    0.993
Extra Trees           0.984     0.991  0.918      0.994   0.988    0.992
LDA                   0.980     0.989  0.908      0.998   0.980    0.985
Random Forest         0.981     0.989  0.896      0.986   0.992    0.985
Gradient Boosting     0.982     0.990  0.908      0.990   0.990    0.994
SGD                   0.984     0.991  0.919      0.996   0.986    0.986
KNN                   0.968     0.982  0.860      0.998   0.966    0.981
Decision Tree         0.970     0.983  0.839      0.982   0.984    0.916
AdaBoost              0.986     0.992  0.935      0.996   0.988    0.993


100%|██████████████████████████████████████████████████████████████████████████████████| 30/30 [00:00<00:00, 31.19it/s]


--- TOPSIS Analizi Başlatılıyor ---

🏆 TOPSIS ANALİZİ BAŞARIYLA TAMAMLANDI 🏆

En iyi 10 model kombinasyonu:

      Classifier  Accuracy  F1-Score    MCC  Precision  Recall  ROC AUC  \
0       AdaBoost     0.986     0.992  0.935      0.996   0.988    0.993   
1       AdaBoost     0.986     0.992  0.928      0.996   0.988    0.994   
2      Log. Reg.     0.986     0.992  0.928      0.996   0.988    0.993   
3      Log. Reg.     0.986     0.992  0.928      0.996   0.988    0.993   
4       AdaBoost     0.986     0.992  0.926      0.994   0.990    0.995   
5    Extra Trees     0.986     0.992  0.926      0.992   0.992    0.986   
6  Random Forest     0.986     0.992  0.922      0.990   0.994    0.996   
7  Random Forest     0.986     0.992  0.922      0.990   0.994    0.996   
8      Log. Reg.     0.984     0.991  0.922      0.996   0.986    0.993   
9            SVM     0.984     0.991  0.920      0.996   0.986    0.990   

         Data Handling Feature Selector Optimization  Topsis_Scor